# 掼蛋·扑克牌识别模型训练（Colab 一键版）

本笔记本在 **Google Colab 免费 GPU** 上训练一个识别扑克牌「数字+花色」的目标检测模型（YOLO）。
训练完会得到一个模型文件，下载回来接到掼蛋 App，就能：摄像头识别 → 自动组牌 → 出牌建议。

**用法**：菜单「运行时 → 更改运行时类型 → 选 GPU」，然后「运行时 → 全部运行」，按提示操作即可。


## 1. 确认已分到 GPU


In [ ]:
!nvidia-smi

## 2. 安装训练框架（Ultralytics YOLO）


In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. 获取扑克牌数据集

推荐用 Roboflow Universe 上公开的「playing cards」数据集（52 类，rank+suit）。
做法：① 免费注册 https://roboflow.com ，在账号设置里拿到 **API Key**；
② 搜索 `playing cards` 数据集，复制它的下载代码；下面是一个示例，把 `YOUR_API_KEY` 换成你的。

> 说明：标准 52 张不含大小王(王)。掼蛋有大小王，可在数据集里另加 2 类「joker_small / joker_big」的标注图，或先只识别 52 张、王用手动补录。


In [ ]:
from roboflow import Roboflow
# 把 YOUR_API_KEY 换成你在 roboflow 账号里的 key；workspace/project/version 用数据集页面给的值
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("roboflow-jvuqo").project("playing-cards-ow27d")  # 示例，按你选的数据集替换
dataset = project.version(4).download('yolov8')
print('数据集下载到:', dataset.location)

## 4. 开始训练

`epochs` 是训练轮数：先用 50 轮试跑，效果不够再加到 100~200。`imgsz` 是输入分辨率。


In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')   # n=最小最快；要更准可换 yolov8s.pt / yolov8m.pt
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50, imgsz=640, batch=16, patience=20,
    project='guandan_cards', name='exp')

## 5. 看看效果（验证集指标 + 预测样张）


In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

## 6. 导出模型（给 App 用）

导出两种格式：`.pt`（Python/服务器用）和 `.onnx`（跨平台、可转手机端 TF.js/TFLite）。


In [ ]:
best = 'guandan_cards/exp/weights/best.pt'
m = YOLO(best)
m.export(format='onnx')   # 生成 best.onnx
print('模型文件：', best, ' 和 同目录的 best.onnx')

## 7. 下载模型到本地


In [ ]:
from google.colab import files
files.download('guandan_cards/exp/weights/best.pt')
files.download('guandan_cards/exp/weights/best.onnx')

## 8. 拿回模型后怎么用

把下载的 `best.pt` 放到本仓库 `vision/` 目录，运行 `python vision/recognize.py 一张牌桌照片.jpg`，
它会输出识别到的牌，并直接调用掼蛋引擎 `decompose/advise` 给出「自动组牌 + 出牌建议」。
手机/眼镜端则用 `best.onnx`（可进一步转 TF.js 在网页里跑、或 TFLite 在手机原生跑）。
